# 07 - Data explorer export for the dashboard

Notebooks 01-02 opened the raw YAML files and built the first score baselines. This notebook reuses that same parsing and cleaning
(plus the leak-safe context from notebooks 03-06) and writes league-wide facts for the dashboard's **Data Explorer**:
season trends, venues, score distribution, run rate by over and era, a season x over heat map and team records.

It needs no model. Run it top to bottom; the last cell writes `explorer.json`.

In [1]:
import os, re, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path(os.environ.get("IPL_PROJECT_ROOT", Path.cwd().parent))
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("YAML files  :", len(list(RAW_DATA_DIR.glob("*.yaml"))))

Project root: c:\Users\amitb\Desktop\ME\Projects\Data_Analysis_Practice\cricket-analytics-lab
YAML files  : 1243


## 1. Parse, clean, context

In [2]:
NOT_A_WICKET = {"retired hurt", "retired not out"}      # not a lost wicket
BOWLER_WICKET_KINDS = {"bowled", "caught", "lbw", "stumped", "caught and bowled", "hit wicket"}   # credited to the bowler


def iter_raw_deliveries(innings_data):
    if "deliveries" in innings_data:                       # layout used by your files
        for entry in innings_data["deliveries"]:
            if isinstance(entry, dict) and entry:
                yield next(iter(entry.values()))
    elif "overs" in innings_data:                          # newer layout
        for over in innings_data["overs"]:
            for d in over.get("deliveries", []):
                yield d
    else:
        raise ValueError("Unknown innings layout: %s" % list(innings_data))


def parse_delivery(d, reg):
    # reg maps player name -> unique id (info.registry.people); falls back to the name itself
    runs = d.get("runs") or {}
    extras = d.get("extras") or {}
    wk = d.get("wickets")
    if wk is None:
        w = d.get("wicket")
        wk = [w] if isinstance(w, dict) else []
    wk = [w for w in wk if w.get("kind") not in NOT_A_WICKET]
    pid = lambda n: reg.get(n, n)
    batter_runs = runs.get("batter", runs.get("batsman", 0)) or 0
    wides, noballs = extras.get("wides", 0) or 0, extras.get("noballs", 0) or 0
    return {
        "total": runs.get("total", 0) or 0, "batter_runs": batter_runs, "wide": wides, "noball": noballs,
        "wickets": len(wk), "out": [pid(w["player_out"]) for w in wk if w.get("player_out")],
        "bowler_wkts": sum(1 for w in wk if w.get("kind") in BOWLER_WICKET_KINDS),
        "bat": pid(d.get("batsman", d.get("batter"))), "bowl": pid(d.get("bowler")),
        "runs_vs_bowler": batter_runs + wides + noballs,          # byes / leg-byes are not charged to the bowler
    }


def summarise_innings(dels):
    # over-end states {k: (runs, wickets, boundaries, dismissed_player_ids)} + per-player batting / bowling lines
    runs = wkts = bnd = legal = bw_wk = 0
    out, states, batters, bowlers = set(), {}, {}, {}
    for d in dels:
        runs += d["total"]; wkts += d["wickets"]; out.update(d["out"]); bw_wk += d["bowler_wkts"]
        if d["batter_runs"] in (4, 6) and d["wide"] == 0:
            bnd += 1
        if d["wide"] == 0:                                        # a ball faced by the batter
            b = batters.setdefault(d["bat"], [0, 0]); b[0] += 1; b[1] += d["batter_runs"]
        bw = bowlers.setdefault(d["bowl"], [0, 0, 0]); bw[1] += d["runs_vs_bowler"]; bw[2] += d["bowler_wkts"]
        if d["wide"] == 0 and d["noball"] == 0:                   # a legal ball
            legal += 1; bw[0] += 1
            if legal % 6 == 0:
                states[legal // 6] = (runs, wkts, bnd, frozenset(out))
    bat_balls = sum(v[0] for v in batters.values())
    return dict(states=states, runs=runs, wkts=wkts, legal=legal, batters=batters, bowlers=bowlers, bowler_wkts=bw_wk,
                bat_rr=(sum(v[1] for v in batters.values()) / bat_balls) if bat_balls else 0.0,       # batter runs per ball faced
                bowl_rr=(sum(v[1] for v in bowlers.values()) / legal) if legal else 0.0)              # runs charged to bowlers per legal ball


def get_innings(match):
    out = []
    for entry in match.get("innings", []):
        if "team" in entry:                                       # newer layout
            if not entry.get("super_over"):
                out.append(entry)
        else:                                                     # layout used by your files: {"1st innings": {...}}
            name, data = next(iter(entry.items()))
            if name in ("1st innings", "2nd innings"):
                out.append(data)
    return out[:2]


def load_all(raw_dir):
    rows, parsed, names, errors = [], {}, {}, []
    for path in sorted(raw_dir.glob("*.yaml")):
        try:
            with path.open("r", encoding="utf-8") as f:
                match = yaml.safe_load(f)
            info = match["info"]
            reg = (info.get("registry") or {}).get("people") or {}
            for n, i in reg.items():
                names[i] = n
            oc = info.get("outcome") or {}
            row = {"match_id": path.stem, "date": pd.to_datetime(str(info["dates"][0])),
                   "venue_raw": info.get("venue"), "scheduled_overs": info.get("overs", 20),
                   "winner": oc.get("winner"), "method": oc.get("method"), "result": oc.get("result")}
            xi = {team: [reg.get(p, p) for p in plist] for team, plist in (info.get("players") or {}).items()}
            inns = get_innings(match)
            if len(inns) == 2:
                s = [summarise_innings([parse_delivery(d, reg) for d in iter_raw_deliveries(x)]) for x in inns]
                row.update(team1=inns[0].get("team"), team2=inns[1].get("team"),
                           final_score=s[0]["runs"], first_wkts=s[0]["wkts"], first_legal=s[0]["legal"],
                           second_total=s[1]["runs"], second_wkts=s[1]["wkts"], second_legal=s[1]["legal"])
                parsed[path.stem] = dict(date=row["date"], xi=xi, inn=s, team1=row["team1"], team2=row["team2"])
            rows.append(row)
        except Exception as exc:
            errors.append((path.name, repr(exc)))
    return pd.DataFrame(rows), parsed, names, errors


raw_matches, parsed, player_names, load_errors = load_all(RAW_DATA_DIR)
print("Matches parsed :", len(raw_matches), "| with both innings:", len(parsed))
print("Players named  :", len(player_names))
print("Load errors    :", len(load_errors), load_errors[:3])

Matches parsed : 1243 | with both innings: 1237
Players named  : 964
Load errors    : 0 []


In [3]:
def status(r):
    if pd.isna(r.get("team2")):
        return "missing an innings"
    if r["scheduled_overs"] != 20:
        return "not a 20-over match"
    if pd.notna(r["method"]):
        return "rain-adjusted target (D/L or similar)"
    if not (r["first_legal"] >= 120 or r["first_wkts"] >= 10):
        return "first innings curtailed"
    if r["result"] in ("tie", "no result") or pd.isna(r["winner"]) or r["second_total"] == r["final_score"]:
        return "tie / no result"
    if (r["second_total"] > r["final_score"]) != (r["winner"] == r["team2"]):
        return "label mismatch (scores vs outcome)"
    return "ok"


raw_matches["status"] = raw_matches.apply(status, axis=1)
print(raw_matches["status"].value_counts().to_string())

matches = raw_matches[raw_matches["status"] == "ok"].copy()
matches["chase_won"] = (matches["second_total"] > matches["final_score"]).astype(int)
matches["target"] = matches["final_score"] + 1
matches["season"] = matches["date"].dt.year

VENUE_ALIASES = {
    "feroz shah kotla": "Arun Jaitley Stadium",
    "sardar patel stadium": "Narendra Modi Stadium",
    "sheikh zayed stadium": "Zayed Cricket Stadium",
    "punjab cricket association stadium": "Punjab Cricket Association IS Bindra Stadium",
    "punjab cricket association is bindra stadium": "Punjab Cricket Association IS Bindra Stadium",
    "subrata roy sahara stadium": "Maharashtra Cricket Association Stadium",
}

def standardize_venue(v):
    if not isinstance(v, str):
        return "Unknown"
    key = re.sub(r"\s+", " ", v.split(",")[0].replace(".", " ")).strip().lower()
    return VENUE_ALIASES.get(key, re.sub(r"\s+", " ", v.split(",")[0].replace(".", " ")).strip())

matches["venue"] = matches["venue_raw"].map(standardize_venue)
print("\nUsable matches:", len(matches), "| venue labels: raw %d -> standardised %d" % (
    matches["venue_raw"].nunique(), matches["venue"].nunique()))

status
ok                                       1183
rain-adjusted target (D/L or similar)      23
tie / no result                            18
first innings curtailed                    13
missing an innings                          6

Usable matches: 1183 | venue labels: raw 60 -> standardised 36


In [4]:
def add_prior_context(df, league_window=60, chase_window=100, venue_shrink=5.0, fallback=160.0):
    m = df.sort_values(["date", "match_id"]).reset_index(drop=True).copy()
    dates = m["date"].to_numpy()
    totals = m["final_score"].to_numpy(dtype=float)
    chase = m["chase_won"].to_numpy(dtype=float)
    venues = m["venue"].to_numpy()
    n_all = len(m)
    league, chase_rate = np.empty(n_all), np.empty(n_all)
    for i in range(n_all):
        prior = np.flatnonzero(dates < dates[i])                 # strictly earlier dates only
        recent = totals[prior][-league_window:]
        league[i] = recent.mean() if len(recent) >= 5 else fallback
        rc = chase[prior][-chase_window:]
        chase_rate[i] = rc.mean() if len(rc) >= 20 else 0.5
    excess = totals - league
    venue_effect, venue_n = np.empty(n_all), np.empty(n_all, dtype=int)
    for i in range(n_all):
        prior = np.flatnonzero(dates < dates[i])
        sv = prior[venues[prior] == venues[i]]
        venue_n[i] = len(sv)
        venue_effect[i] = excess[sv].sum() / (len(sv) + venue_shrink)
    m["league_recent"], m["excess"] = league, excess
    m["venue_effect"], m["venue_n"], m["chase_rate_recent"] = venue_effect, venue_n, chase_rate
    return m


matches = add_prior_context(matches)
TOTAL_OVERS = 20

matches[['venue', 'final_score', 'target', 'chase_won', 'league_recent', 'venue_effect']].tail(3).round(2)

,venue,final_score,target,chase_won,league_recent,venue_effect
1180,Maharaja Yadavindra Singh International Cricke...,243.0,244.0,0,197.45,-6.41
1181,Maharaja Yadavindra Singh International Cricke...,214.0,215.0,1,198.80,-3.94
1182,Narendra Modi Stadium,155.0,156.0,1,198.87,2.01


## 2. Export

In [5]:
# ===== Dashboard export =====================================================================
# Writes JSON files that the React dashboard reads. Safe to re-run; it only writes files.
import json, datetime

DASH_DIR = Path(os.environ.get("IPL_DASHBOARD_DATA", PROJECT_ROOT / "dashboard" / "public" / "data"))
DASH_DIR.mkdir(parents=True, exist_ok=True)


def _clean(x):
    if isinstance(x, dict):
        return {str(k): _clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_clean(v) for v in x]
    if isinstance(x, pd.DataFrame):
        return _clean(x.reset_index().to_dict("records"))
    if isinstance(x, pd.Series):
        return _clean(x.to_dict())
    if isinstance(x, np.ndarray):
        return _clean(x.tolist())
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return round(float(x), 5) if np.isfinite(x) else None
    if isinstance(x, (pd.Timestamp, datetime.date)):
        return str(x)[:10]
    if x is None or isinstance(x, str):
        return x
    return str(x)


def tbl(df, scale=1.0):
    # pivot-style table -> {"index": [...], "columns": [...], "data": [[...]]}
    d = df.astype(float) * scale
    return {"index": _clean(list(d.index)), "columns": [str(c) for c in d.columns], "data": _clean(d.values.tolist())}


def recs(df):
    d = df.copy()
    for c in d.columns:
        if str(d[c].dtype).startswith("category") or d[c].dtype == object:
            d[c] = d[c].astype(str)
    return _clean(d.reset_index(drop=True).to_dict("records"))


def dash_save(name, payload):
    meta = {"notebook": name, "generated": datetime.datetime.now().isoformat(timespec="seconds"),
            "matches": int(len(matches)), "first_season": int(matches["season"].min()), "last_season": int(matches["season"].max())}
    path = DASH_DIR / (name + ".json")
    path.write_text(json.dumps(_clean({"meta": meta, **payload}), separators=(",", ":")), encoding="utf-8")
    print("wrote %s (%.0f KB)" % (path, path.stat().st_size / 1024))

ERA_NAMES = {"Delhi Daredevils": "Delhi Capitals", "Kings XI Punjab": "Punjab Kings",
             "Royal Challengers Bangalore": "Royal Challengers Bengaluru", "Rising Pune Supergiants": "Rising Pune Supergiant"}
m = matches.copy()
m["team1"] = m["team1"].replace(ERA_NAMES); m["team2"] = m["team2"].replace(ERA_NAMES)
m["first_rr"] = m["final_score"] / (m["first_legal"] / 6)
m["second_rr"] = m["second_total"] / (m["second_legal"] / 6)

seasons_tbl = m.groupby("season").agg(matches=("match_id", "size"), avg_first=("final_score", "mean"), avg_second=("second_total", "mean"),
                                      chase_win=("chase_won", "mean"), first_rr=("first_rr", "mean"), second_rr=("second_rr", "mean"),
                                      high=("final_score", "max"), low=("final_score", "min")).reset_index()
venue_tbl = (m.groupby("venue").agg(matches=("match_id", "size"), avg_first=("final_score", "mean"), chase_win=("chase_won", "mean"),
                                    excess=("excess", "sum")).reset_index())
venue_tbl["venue_effect"] = venue_tbl["excess"] / (venue_tbl["matches"] + 5.0)
venue_tbl = venue_tbl.drop(columns="excess").sort_values("matches", ascending=False).head(30)

edges = list(range(80, 281, 10))
hist, _ = np.histogram(m["final_score"], bins=edges)
score_hist = [{"from": edges[i], "to": edges[i + 1], "matches": int(hist[i])} for i in range(len(hist))]

ERAS = [("2008-12", 2008, 2012), ("2013-17", 2013, 2017), ("2018-22", 2018, 2022), ("2023+", 2023, 2100)]
def per_over(idx, season_lo, season_hi):
    runs_o, wk_o = np.zeros(20), np.zeros(20); n = 0
    for r in m[(m["season"] >= season_lo) & (m["season"] <= season_hi)].itertuples():
        st = parsed[r.match_id]["inn"][idx]["states"]
        if 20 not in st:
            continue
        prev_r = prev_w = 0
        for k in range(1, 21):
            r_, w_ = st[k][:2]
            runs_o[k - 1] += r_ - prev_r; wk_o[k - 1] += w_ - prev_w; prev_r, prev_w = r_, w_
        n += 1
    return {"matches": n, "runs": (runs_o / max(n, 1)).round(3).tolist(), "wickets": (wk_o / max(n, 1)).round(3).tolist()}
over_profile = {name: {"first": per_over(0, lo, hi), "second": per_over(1, lo, hi)} for name, lo, hi in ERAS}

heat = []
for s_, g_ in m.groupby("season"):
    acc = np.zeros(20); n = 0
    for r in g_.itertuples():
        st = parsed[r.match_id]["inn"][0]["states"]
        if 20 not in st:
            continue
        prev = 0
        for k in range(1, 21):
            acc[k - 1] += st[k][0] - prev; prev = st[k][0]
        n += 1
    heat.append({"season": int(s_), "runs_by_over": (acc / max(n, 1)).round(2).tolist()})

teams = {}
for t in sorted(set(m["team1"]) | set(m["team2"])):
    bf, ch = m[m["team1"] == t], m[m["team2"] == t]
    wins_bf, wins_ch = int((bf["chase_won"] == 0).sum()), int(ch["chase_won"].sum())
    teams[t] = dict(team=t, matches=len(bf) + len(ch), wins=wins_bf + wins_ch, bat_first=len(bf), bat_first_wins=wins_bf, chasing=len(ch), chasing_wins=wins_ch)
teams_tbl = sorted(teams.values(), key=lambda r: -r["matches"])

top_scores = m.sort_values("final_score", ascending=False).head(10)[["date", "team1", "team2", "venue", "final_score", "second_total"]]
payload = {"seasons": recs(seasons_tbl), "venues": recs(venue_tbl), "score_hist": score_hist, "over_profile": over_profile,
           "season_over_heat": heat, "teams": teams_tbl, "top_scores": recs(top_scores),
           "funnel": {str(k): int(v) for k, v in raw_matches["status"].value_counts().items()}, "files": int(len(raw_matches))}
dash_save("explorer", payload)

wrote c:\Users\amitb\Desktop\ME\Projects\Data_Analysis_Practice\cricket-analytics-lab\dashboard\public\data\explorer.json (15 KB)
